In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import cv2

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df = pd.read_csv('/content/drive/MyDrive/monkeytype/monkeytyping.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6159 entries, 0 to 6158
Data columns (total 31 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   participant_id             6159 non-null   object 
 1   date                       6159 non-null   object 
 2   original_filename          6159 non-null   object 
 3   trial_number               6159 non-null   int64  
 4   reaction_time              6159 non-null   object 
 5   correct                    6159 non-null   int64  
 6   correct_position           6159 non-null   object 
 7   chosen_position            6159 non-null   object 
 8   prompt                     0 non-null      float64
 9   sample_name                0 non-null      float64
 10  file_1_name                6159 non-null   object 
 11  file_2_name                6159 non-null   object 
 12  file_3_name                6139 non-null   object 
 13  file_4_name                6139 non-null   objec

In [4]:
df_symbols = df[df['colored'].isna()].copy(deep=True)
df_symbols = df_symbols[df_symbols['file_3_name'].notna()].copy(deep=True)
df_symbols = df_symbols[df_symbols['chosen_position']!='No decision'].copy(deep=True)
#df_symbols.info()
image_filepath_main = '/content/drive/MyDrive/monkeytype/emnist_rare_ESRGAN_improved/'
file_cols_name = ['file_1_name', 'file_2_name', 'file_3_name', 'file_4_name']
for f in file_cols_name:
  df_symbols[f] = df_symbols[f].astype(str)
  for index, row in df_symbols.iterrows():
    df_symbols.loc[index, f] = df_symbols.loc[index, f][0] + '/'+df_symbols.loc[index, f][2:] + '_out.png'
print(df_symbols[file_cols_name[0]])

0       y/00902_out.png
1       s/00320_out.png
2       c/01270_out.png
3       f/00745_out.png
4       s/01180_out.png
             ...       
6154    f/01252_out.png
6155    m/01187_out.png
6156    m/00439_out.png
6157    m/00413_out.png
6158    c/00117_out.png
Name: file_1_name, Length: 4675, dtype: object


In [5]:
df_files = df_symbols[file_cols_name].copy(deep = True)
df_files.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4675 entries, 0 to 6158
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   file_1_name  4675 non-null   object
 1   file_2_name  4675 non-null   object
 2   file_3_name  4675 non-null   object
 3   file_4_name  4675 non-null   object
dtypes: object(4)
memory usage: 311.7+ KB


In [6]:
symbols_list = ['m', 'c', 's', 'y', 'f', 'j']
new_file_cols = ['file_1_symb', 'file_2_symb', 'file_3_symb', 'file_4_symb']
for index, row in df_files.iterrows():
  for f, fs in zip(file_cols_name, new_file_cols):
    df_files.loc[index, fs] = df_files.loc[index, f][0]
df_files.head()

,file_1_name,file_2_name,file_3_name,file_4_name,file_1_symb,file_2_symb,file_3_symb,file_4_symb
0,y/00902_out.png,c/00891_out.png,f/01542_out.png,m/00280_out.png,y,c,f,m
1,s/00320_out.png,f/01788_out.png,m/01766_out.png,j/01080_out.png,s,f,m,j
2,c/01270_out.png,y/00523_out.png,j/01415_out.png,m/00902_out.png,c,y,j,m
3,f/00745_out.png,s/00125_out.png,m/01681_out.png,c/00697_out.png,f,s,m,c
4,s/01180_out.png,y/00443_out.png,j/01331_out.png,m/00367_out.png,s,y,j,m


In [7]:
for index, row in df_files.iterrows():
  for f in file_cols_name:
    df_files.loc[index, f] = image_filepath_main + df_files.loc[index, f]
df_files.head()

,file_1_name,file_2_name,file_3_name,file_4_name,file_1_symb,file_2_symb,file_3_symb,file_4_symb
0,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,y,c,f,m
1,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,s,f,m,j
2,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,c,y,j,m
3,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,f,s,m,c
4,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,/content/drive/MyDrive/monkeytype/emnist_rare_...,s,y,j,m


In [8]:
brightness_columns = ['brightness0', 'brightness1', 'brightness2', 'brightness3']
local_symbol_columns = [['local_symbol_cls00', 'local_symbol_cls01', 'local_symbol_cls02', 'local_symbol_cls03', 'local_symbol_cls04', 'local_symbol_cls05'],
['local_symbol_cls10', 'local_symbol_cls11', 'local_symbol_cls12', 'local_symbol_cls13', 'local_symbol_cls14', 'local_symbol_cls15'],
['local_symbol_cls20', 'local_symbol_cls21', 'local_symbol_cls22', 'local_symbol_cls23', 'local_symbol_cls24', 'local_symbol_cls25'],
['local_symbol_cls30', 'local_symbol_cls31', 'local_symbol_cls32', 'local_symbol_cls33', 'local_symbol_cls34', 'local_symbol_cls35']]
correct_symbol_columns = ['correct_symbol_cls0','correct_symbol_cls1','correct_symbol_cls2','correct_symbol_cls3','correct_symbol_cls4','correct_symbol_cls5']
dataset_columns = ['participant_id',
                   'brightness0', 'brightness1', 'brightness2', 'brightness3',
                   'local_symbol_cls00', 'local_symbol_cls01', 'local_symbol_cls02', 'local_symbol_cls03', 'local_symbol_cls04', 'local_symbol_cls05',
                   'local_symbol_cls10', 'local_symbol_cls11', 'local_symbol_cls12', 'local_symbol_cls13', 'local_symbol_cls14', 'local_symbol_cls15',
                   'local_symbol_cls20', 'local_symbol_cls21', 'local_symbol_cls22', 'local_symbol_cls23', 'local_symbol_cls24', 'local_symbol_cls25',
                   'local_symbol_cls30', 'local_symbol_cls31', 'local_symbol_cls32', 'local_symbol_cls33', 'local_symbol_cls34', 'local_symbol_cls35',
                   'correct_symbol_cls0','correct_symbol_cls1','correct_symbol_cls2','correct_symbol_cls3','correct_symbol_cls4','correct_symbol_cls5',
                   'correct_local_cls',
                   'participant_linspace',
                   'time_reaction_task',
                   #monkey profile
                   #'bh0', 'bh1', 'bh2', 'bh3', 'bh4', 'bh5', 'bh6', 'bh7', 'bh8', 'bh9',
                   #'lh0', 'lh1', 'lh2', 'lh3',
                   #'sh0', 'sh1', 'sh2', 'sh3', 'sh4', 'sh5',
                   'time_reaction_mean',
                   'chosen_local_cls']
df_dataset = pd.DataFrame(columns = dataset_columns)
df_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 40 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   participant_id        0 non-null      object
 1   brightness0           0 non-null      object
 2   brightness1           0 non-null      object
 3   brightness2           0 non-null      object
 4   brightness3           0 non-null      object
 5   local_symbol_cls00    0 non-null      object
 6   local_symbol_cls01    0 non-null      object
 7   local_symbol_cls02    0 non-null      object
 8   local_symbol_cls03    0 non-null      object
 9   local_symbol_cls04    0 non-null      object
 10  local_symbol_cls05    0 non-null      object
 11  local_symbol_cls10    0 non-null      object
 12  local_symbol_cls11    0 non-null      object
 13  local_symbol_cls12    0 non-null      object
 14  local_symbol_cls13    0 non-null      object
 15  local_symbol_cls14    0 non-null      object
 16  lo

In [15]:
for index, row in df_symbols.iterrows():
  df_dataset.loc[index, 'participant_id'] = df_symbols.loc[index, 'participant_id']
  df_dataset.loc[index, 'chosen_local_cls'] = int(df_symbols.loc[index, 'chosen_position'][0])-1
  df_dataset.loc[index, 'correct_local_cls'] = int(df_symbols.loc[index, 'correct_position'][0])-1
  df_dataset.loc[index, 'time_reaction_task'] = df_symbols.loc[index, 'reaction_time']
  df_dataset.loc[index, 'time_reaction_mean'] = df_symbols.loc[index, 'mean_reaction_time']
  #symbol_cls_correct = df_symbols.loc[index, 'correct_position'][0]
  for f, fs, bc in zip(file_cols_name, new_file_cols, brightness_columns):
    img = cv2.imread(df_files.loc[index, f], cv2.IMREAD_GRAYSCALE)
    if img is None:
      print(f"Ошибка: не удалось загрузить изображение по пути: {df_files.loc[index, f]}")
    df_dataset.loc[index, bc] = np.mean(img)
  #for f, fs, sc in zip(file_cols_name, new_file_cols, symbol_columns):
  #  symbol = str(df_files.loc[index, fs])
  #  df_dataset.loc[index, sc] = symbols_list.index(symbol)
  #  if symbol_cls_correct in fs:
  #    df_dataset.loc[index, 'correct_symbol_cls'] = symbols_list.index(symbol)
  if index%100==0:
    print(index)


0
100
400
500
600
700
1000
1100
1200
1300
1400
1500
1700
2000
2200
2300
2400
2500
2600
2700
2800
2900
3100
3200
3300
3400
3600
3700
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100


In [16]:
for index, row in df_symbols.iterrows():
  symbol_cls_correct = df_symbols.loc[index, 'correct_position'][0]
  for fs, lscs in zip(new_file_cols, local_symbol_columns):
    symbol = str(df_files.loc[index, fs])
    symbol_ind = symbols_list.index(symbol)
    df_dataset.loc[index, lscs[symbol_ind]] = 1
    if symbol_cls_correct in fs:
      df_dataset.loc[index, correct_symbol_columns[symbol_ind]] = 1
df_dataset.fillna(0, inplace=True)

/tmp/ipython-input-3330132409.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_dataset.fillna(0, inplace=True)


In [17]:
df_dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4675 entries, 0 to 6158
Data columns (total 40 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   participant_id        4675 non-null   object 
 1   brightness0           4675 non-null   float64
 2   brightness1           4675 non-null   float64
 3   brightness2           4675 non-null   float64
 4   brightness3           4675 non-null   float64
 5   local_symbol_cls00    4675 non-null   int64  
 6   local_symbol_cls01    4675 non-null   int64  
 7   local_symbol_cls02    4675 non-null   int64  
 8   local_symbol_cls03    4675 non-null   int64  
 9   local_symbol_cls04    4675 non-null   int64  
 10  local_symbol_cls05    4675 non-null   int64  
 11  local_symbol_cls10    4675 non-null   int64  
 12  local_symbol_cls11    4675 non-null   int64  
 13  local_symbol_cls12    4675 non-null   int64  
 14  local_symbol_cls13    4675 non-null   int64  
 15  local_symbol_cls14    4675

In [19]:
g = df_symbols.groupby('participant_id')
pos = g.cumcount()

# размер группы (кол-во строк в сессии), "растянутый" до длины df
cnt = g['reaction_time'].transform('size')  # any column, просто чтобы вызвать transform

# чтобы получить [0, 1] включительно: pos / (cnt-1)
# аккуратно обходим деление на 0, если в группе всего одна строка
denom = (cnt - 1).where(cnt > 1, 1)  # если cnt==1, делим на 1 (получится 0)

df_dataset['participant_linspace'] = pos / denom

df_dataset

,participant_id,brightness0,brightness1,brightness2,brightness3,local_symbol_cls00,local_symbol_cls01,local_symbol_cls02,local_symbol_cls03,local_symbol_cls04,...,correct_symbol_cls1,correct_symbol_cls2,correct_symbol_cls3,correct_symbol_cls4,correct_symbol_cls5,correct_local_cls,participant_linspace,time_reaction_task,time_reaction_mean,chosen_local_cls
0,Jupiter,46.317761,35.222178,35.280851,52.363520,0,0,0,1,0,...,1,0,0,0,0,1,0.000000,1029,980.0,2
1,Jupiter,56.269372,30.205277,45.111926,39.408004,0,0,1,0,0,...,0,0,0,1,0,1,0.000593,579,980.0,1
2,Jupiter,55.413265,31.938776,35.726483,50.945871,0,1,0,0,0,...,0,0,0,0,1,2,0.001186,688,980.0,3
3,Jupiter,33.661432,37.567443,28.843033,48.010443,0,0,0,0,1,...,1,0,0,0,0,3,0.001779,578,980.0,1
4,Jupiter,64.207270,43.166454,26.046397,34.599649,0,0,1,0,0,...,0,0,0,0,0,3,0.002372,848,980.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6154,Yunt,45.730469,43.042331,15.456154,64.904735,0,0,0,0,1,...,0,0,1,0,0,1,0.997557,1177,1424.0,1
6155,Yunt,38.996492,29.708785,30.469707,59.547832,1,0,0,0,0,...,0,0,0,0,0,0,0.998167,1132,1424.0,3
6156,Yunt,41.052774,59.851004,36.981904,43.856505,1,0,0,0,0,...,0,1,0,0,0,1,0.998778,1369,1424.0,2
6157,Yunt,54.730070,31.151786,29.944276,38.622290,1,0,0,0,0,...,0,0,1,0,0,1,0.999389,1623,1424.0,3


In [10]:
df_Jupiter = df_symbols[df_symbols['participant_id']=='Jupiter'].copy(deep=True)
df_Jupiter['datetime'] = pd.to_datetime(df_Jupiter['datetime'], format='%d.%m.%Y %H:%M:%S')
df_Feliks = df_symbols[df_symbols['participant_id']=='Feliks'].copy(deep=True)
df_Feliks['datetime'] = pd.to_datetime(df_Feliks['datetime'], format='%d.%m.%Y %H:%M:%S')
df_Yunt = df_symbols[df_symbols['participant_id']=='Yunt'].copy(deep=True)
df_Yunt['datetime'] = pd.to_datetime(df_Yunt['datetime'], format='%d.%m.%Y %H:%M:%S')

In [ ]:
values_J = df_dataset[df_dataset['participant_id']=='Jupiter']['chosen_local_cls'].value_counts(normalize=True)
df_values_J = values_J.sort_index().to_frame().T

df_dataset.loc[df_dataset['participant_id']=='Jupiter', ['lh0', 'lh1', 'lh2', 'lh3']] = df_values_J.values[0]
values_F = df_dataset[df_dataset['participant_id']=='Feliks']['chosen_local_cls'].value_counts(normalize=True)
df_values_F = values_F.sort_index().to_frame().T

df_dataset.loc[df_dataset['participant_id']=='Feliks', ['lh0', 'lh1', 'lh2', 'lh3']] = df_values_F.values[0]
values_Y = df_dataset[df_dataset['participant_id']=='Yunt']['chosen_local_cls'].value_counts(normalize=True)
df_values_Y = values_F.sort_index().to_frame().T

df_dataset.loc[df_dataset['participant_id']=='Yunt', ['lh0', 'lh1', 'lh2', 'lh3']] = df_values_Y.values[0]

,lh0,lh1,lh2,lh3
0,0.17131,0.508002,0.260225,0.060462
1,0.17131,0.508002,0.260225,0.060462
2,0.17131,0.508002,0.260225,0.060462
3,0.17131,0.508002,0.260225,0.060462
4,0.17131,0.508002,0.260225,0.060462
...,...,...,...,...
6154,0.085185,0.405185,0.295556,0.214074
6155,0.085185,0.405185,0.295556,0.214074
6156,0.085185,0.405185,0.295556,0.214074
6157,0.085185,0.405185,0.295556,0.214074


In [ ]:
n_rows = 1
df_sym_hist_Yunt = pd.DataFrame(np.zeros((n_rows, len(symbols_list))),columns=symbols_list)
for index, row in df_Yunt.iterrows():
  f = ''
  index_ch = df_Yunt.loc[index, 'chosen_position'][0]
  if index_ch!="N":
    for i, s in enumerate(file_cols_name):
      if index_ch in s:
        f = file_cols_name[i]
    symbol2count = df_Yunt.loc[index, f][0]
    df_sym_hist_Yunt.loc[0, symbol2count]+=1.0
row = df_sym_hist_Yunt.iloc[0]
df_sym_hist_Yunt.iloc[0] = row / row.sum()

df_sym_hist_Jupiter = pd.DataFrame(np.zeros((n_rows, len(symbols_list))),columns=symbols_list)
for index, row in df_Jupiter.iterrows():
  f = ''
  index_ch = df_Jupiter.loc[index, 'chosen_position'][0]
  if index_ch!="N":
    for i, s in enumerate(file_cols_name):
      if index_ch in s:
        f = file_cols_name[i]
    symbol2count = df_Jupiter.loc[index, f][0]
    df_sym_hist_Jupiter.loc[0, symbol2count]+=1.0
row = df_sym_hist_Jupiter.iloc[0]
df_sym_hist_Jupiter.iloc[0] = row / row.sum()

df_sym_hist_Feliks = pd.DataFrame(np.zeros((n_rows, len(symbols_list))),columns=symbols_list)
for index, row in df_Feliks.iterrows():
  f = ''
  index_ch = df_Feliks.loc[index, 'chosen_position'][0]
  if index_ch!="N":
    for i, s in enumerate(file_cols_name):
      if index_ch in s:
        f = file_cols_name[i]
    symbol2count = df_Feliks.loc[index, f][0]
    df_sym_hist_Feliks.loc[0, symbol2count]+=1.0
row = df_sym_hist_Feliks.iloc[0]
df_sym_hist_Feliks.iloc[0] = row / row.sum()

In [ ]:
df_dataset.loc[df_dataset['participant_id']=='Jupiter', ['sh0', 'sh1', 'sh2', 'sh3', 'sh4', 'sh5']] = df_sym_hist_Jupiter.values[0]
df_dataset.loc[df_dataset['participant_id']=='Feliks', ['sh0', 'sh1', 'sh2', 'sh3', 'sh4', 'sh5']] = df_sym_hist_Feliks.values[0]
df_dataset.loc[df_dataset['participant_id']=='Yunt', ['sh0', 'sh1', 'sh2', 'sh3', 'sh4', 'sh5']] = df_sym_hist_Yunt.values[0]

df_dataset

,participant_id,brightness0,brightness1,brightness2,brightness3,symbol_cls0,symbol_cls1,symbol_cls2,symbol_cls3,correct_symbol_cls,...,lh2,lh3,sh0,sh1,sh2,sh3,sh4,sh5,time_reaction_mean,chosen_local_cls
0,Jupiter,46.317761,35.222178,35.280851,52.36352,3,1,4,0,1,...,0.260225,0.060462,0.116183,0.191464,0.190871,0.239478,0.180794,0.081209,980.0,2
1,Jupiter,56.269372,30.205277,45.111926,39.408004,2,4,0,5,4,...,0.260225,0.060462,0.116183,0.191464,0.190871,0.239478,0.180794,0.081209,980.0,1
2,Jupiter,55.413265,31.938776,35.726483,50.945871,1,3,5,0,5,...,0.260225,0.060462,0.116183,0.191464,0.190871,0.239478,0.180794,0.081209,980.0,3
3,Jupiter,33.661432,37.567443,28.843033,48.010443,4,2,0,1,1,...,0.260225,0.060462,0.116183,0.191464,0.190871,0.239478,0.180794,0.081209,980.0,1
4,Jupiter,64.20727,43.166454,26.046397,34.599649,2,3,5,0,0,...,0.260225,0.060462,0.116183,0.191464,0.190871,0.239478,0.180794,0.081209,980.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6154,Yunt,45.730469,43.042331,15.456154,64.904735,4,3,5,1,3,...,0.295556,0.214074,0.142247,0.189866,0.200244,0.239316,0.140415,0.087912,1424.0,1
6155,Yunt,38.996492,29.708785,30.469707,59.547832,0,1,3,2,0,...,0.295556,0.214074,0.142247,0.189866,0.200244,0.239316,0.140415,0.087912,1424.0,3
6156,Yunt,41.052774,59.851004,36.981904,43.856505,0,2,5,1,2,...,0.295556,0.214074,0.142247,0.189866,0.200244,0.239316,0.140415,0.087912,1424.0,2
6157,Yunt,54.73007,31.151786,29.944276,38.62229,0,3,5,1,3,...,0.295556,0.214074,0.142247,0.189866,0.200244,0.239316,0.140415,0.087912,1424.0,3


In [ ]:
hist_bright_Jupiter = [0.0166073546856465, 0.09193357058125741, 0.2609727164887307, 0.2858837485172005, 0.20106761565836298, 0.10498220640569395, 0.028469750889679714, 0.009489916963226572, 0.0005931198102016608, 0.0]
hist_bright_Feliks = [0.01112759643916914, 0.052670623145400594, 0.14836795252225518, 0.22032640949554896, 0.23516320474777447, 0.1795252225519288, 0.09272997032640949, 0.04080118694362018, 0.015578635014836795, 0.00370919881305638]
hist_bright_Yunt = [0.004276114844227245, 0.05070250458155162, 0.16371411117898596, 0.2596212583995113, 0.2516799022602321, 0.14599877825290164, 0.0824679291386683, 0.030543677458766034, 0.007330482590103849, 0.0036652412950519244]

df_dataset.loc[df_dataset['participant_id']=='Jupiter', ['bh0', 'bh1', 'bh2', 'bh3', 'bh4', 'bh5', 'bh6', 'bh7', 'bh8', 'bh9']] = hist_bright_Jupiter
df_dataset.loc[df_dataset['participant_id']=='Feliks', ['bh0', 'bh1', 'bh2', 'bh3', 'bh4', 'bh5', 'bh6', 'bh7', 'bh8', 'bh9']] = hist_bright_Feliks
df_dataset.loc[df_dataset['participant_id']=='Yunt', ['bh0', 'bh1', 'bh2', 'bh3', 'bh4', 'bh5', 'bh6', 'bh7', 'bh8', 'bh9']] = hist_bright_Yunt

In [20]:
df_dataset.to_csv('/content/drive/MyDrive/monkeytype/mt_dataset_tm.csv', index=False)

In [21]:
df_dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4675 entries, 0 to 6158
Data columns (total 40 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   participant_id        4675 non-null   object 
 1   brightness0           4675 non-null   float64
 2   brightness1           4675 non-null   float64
 3   brightness2           4675 non-null   float64
 4   brightness3           4675 non-null   float64
 5   local_symbol_cls00    4675 non-null   int64  
 6   local_symbol_cls01    4675 non-null   int64  
 7   local_symbol_cls02    4675 non-null   int64  
 8   local_symbol_cls03    4675 non-null   int64  
 9   local_symbol_cls04    4675 non-null   int64  
 10  local_symbol_cls05    4675 non-null   int64  
 11  local_symbol_cls10    4675 non-null   int64  
 12  local_symbol_cls11    4675 non-null   int64  
 13  local_symbol_cls12    4675 non-null   int64  
 14  local_symbol_cls13    4675 non-null   int64  
 15  local_symbol_cls14    4675